In [1]:
pip install dask[complete] numpy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [2]:
# Membaca dataset
import pandas as pd
df = pd.read_csv("Dataset_Ojol_Transaction.csv")
print(df.head())
print(df.info())
print("Dataset shape:", df.shape)

     id                                   date  mode  \
0  1617  2019/03/09 20:45 s/d 2019/03/09 19:55  BIKE   
1  1297  2019/03/09 19:55 s/d 2019/03/09 19:54  FOOD   
2  1394  2019/03/09 19:54 s/d 2019/03/09 18:56  SHOP   
3  1120  2019/03/09 18:56 s/d 2019/03/09 12:28  FOOD   
4  2053  2019/03/09 12:28 s/d 2019/03/08 18:25   CAR   

                                         from_alamat   from_kelurahan  \
0                    Gang Ikhwan No.16,  Sungai Jawi      DARAT SEKIP   
1  Neo Shabu-Shabu Steak & Shake, Johar, Jl. Joha...  SUNGAI BANGKONG   
2             Alfamart Pontianak Mall, Jl Teuku Umar      DARAT SEKIP   
3  Parklife, Jl. Karimata No.64, Sungai Bangkong,...          MARIANA   
4        Jl. Tabrani Ahmad No.12,  Sungai Jawi Dalam         PAL LIMA   

    from_kecamatan               from_latlng  \
0   PONTIANAK KOTA     -0,0303277,109,297753   
1   PONTIANAK KOTA       -0,02861,109,329253   
2   PONTIANAK KOTA    -0,0301863,109,3356331   
3   PONTIANAK KOTA    -0,0305815

In [3]:
# EDA
print("\nDeskripsi Statistik:")
print(df.describe(include='all'))


Deskripsi Statistik:
                 id                                   date  mode  \
count   1017.000000                                   1017  1017   
unique          NaN                                   1017     4   
top             NaN  2018/09/10 09:12 s/d 1900/01/00 00:00  BIKE   
freq            NaN                                      1   325   
mean    1621.648968                                    NaN   NaN   
std      296.954571                                    NaN   NaN   
min     1101.000000                                    NaN   NaN   
25%     1367.000000                                    NaN   NaN   
50%     1624.000000                                    NaN   NaN   
75%     1878.000000                                    NaN   NaN   
max     2132.000000                                    NaN   NaN   

                  from_alamat from_kelurahan  from_kecamatan  \
count                    1017           1017            1017   
unique                    504    

In [4]:
# Cek missing values
print("\nJumlah Missing Values Tiap Kolom:")
print(df.isnull().sum())


Jumlah Missing Values Tiap Kolom:
id                            0
date                          0
mode                          0
from_alamat                   0
from_kelurahan                0
from_kecamatan                0
from_latlng                   0
to_alamat                     3
to_kelurahan                  0
to_kecamatan                  0
to_latlng                     0
distance                      0
amount_delivery               0
amount_merchant               0
transaction_amount_total      0
customer_id                   0
customer_gender               0
customer_birthdate            0
driver_id                     0
driver_gender                 0
driver_birthdate              0
kendaraan_jenis               0
kendaraan_merk                0
merchant_id                 567
merchant_name               567
merchant_category           567
dtype: int64


In [5]:
# Handling Missing Values
# Identifikasi Kolom Numerik & Kategorikal
num_cols = df.select_dtypes(include=["number"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

# Cek missing values per kolom
missing = df.isnull().sum()
print("\nJumlah missing values per kolom:\n", missing)

# Isi missing value numerik dengan mean per kolom
for col in num_cols:
    mean_val = df[col].mean()
    df[col] = df[col].fillna(mean_val)

# Isi missing value kategorikal dengan "Unknown"
for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

# Cek ulang setelah handling
print("\nMissing values setelah di-handle:\n", df.isnull().sum())


Jumlah missing values per kolom:
 id                            0
date                          0
mode                          0
from_alamat                   0
from_kelurahan                0
from_kecamatan                0
from_latlng                   0
to_alamat                     3
to_kelurahan                  0
to_kecamatan                  0
to_latlng                     0
distance                      0
amount_delivery               0
amount_merchant               0
transaction_amount_total      0
customer_id                   0
customer_gender               0
customer_birthdate            0
driver_id                     0
driver_gender                 0
driver_birthdate              0
kendaraan_jenis               0
kendaraan_merk                0
merchant_id                 567
merchant_name               567
merchant_category           567
dtype: int64

Missing values setelah di-handle:
 id                          0
date                        0
mode                    

In [6]:
# Encoding kolom customer_gender
if "customer_gender" in df.columns:
    df["customer_gender_encoded"] = df["customer_gender"].map({
        "l": 0,        
        "p": 1,        
        "x": 2,        
        "unknown": -1  
    })

# Encoding kolom kendaraan_jenis
if "kendaraan_jenis" in df.columns:
    df["kendaraan_jenis_encoded"] = df["kendaraan_jenis"].map({
        "motor": 0,
        "mobil": 1,
        "unknown": -1
    })

# Tampilkan hasil cek
print("\nContoh hasil encoding:")
print(df[["customer_gender", "customer_gender_encoded", 
          "kendaraan_jenis", "kendaraan_jenis_encoded"]].head(10))


Contoh hasil encoding:
  customer_gender  customer_gender_encoded kendaraan_jenis  \
0               P                      NaN           MOTOR   
1               L                      NaN           MOTOR   
2               L                      NaN           MOTOR   
3               L                      NaN           MOTOR   
4               L                      NaN           MOBIL   
5               P                      NaN           MOTOR   
6               P                      NaN           MOTOR   
7               P                      NaN           MOBIL   
8               L                      NaN           MOBIL   
9               L                      NaN           MOTOR   

   kendaraan_jenis_encoded  
0                      NaN  
1                      NaN  
2                      NaN  
3                      NaN  
4                      NaN  
5                      NaN  
6                      NaN  
7                      NaN  
8                      NaN  
9  

In [7]:
# Rapikan tanggal 

# Pisahkan start_time & end_time
df[["start_time", "end_time"]] = df["date"].str.split(" s/d ", expand=True)

# Ubah ke datetime
df["start_time"] = pd.to_datetime(df["start_time"], format="%Y/%m/%d %H:%M", errors="coerce")
df["end_time"] = pd.to_datetime(df["end_time"], format="%Y/%m/%d %H:%M", errors="coerce")

# Fix kalau end_time < start_time (tukar posisi)
mask = df["end_time"] < df["start_time"]
df.loc[mask, ["start_time", "end_time"]] = df.loc[mask, ["end_time", "start_time"]].values

# Hitung durasi dalam menit
df["duration_minutes"] = (df["end_time"] - df["start_time"]).dt.total_seconds() / 60

# Tampilkan hasil
print(df[["id", "start_time", "end_time", "duration_minutes"]].head(10))

     id          start_time            end_time  duration_minutes
0  1617 2019-03-09 19:55:00 2019-03-09 20:45:00              50.0
1  1297 2019-03-09 19:54:00 2019-03-09 19:55:00               1.0
2  1394 2019-03-09 18:56:00 2019-03-09 19:54:00              58.0
3  1120 2019-03-09 12:28:00 2019-03-09 18:56:00             388.0
4  2053 2019-03-08 18:25:00 2019-03-09 12:28:00            1083.0
5  1574 2019-03-08 18:25:00 2019-03-08 18:25:00               0.0
6  1882 2019-03-08 12:58:00 2019-03-08 18:25:00             327.0
7  1958 2019-03-07 18:51:00 2019-03-08 12:58:00            1087.0
8  2009 2019-03-07 16:30:00 2019-03-07 18:51:00             141.0
9  1671 2019-03-07 15:53:00 2019-03-07 16:30:00              37.0


In [8]:
# Feature extraction
df["year"] = df["start_time"].dt.year
df["month"] = df["start_time"].dt.month
df["day"] = df["start_time"].dt.day
df["weekday"] = df["start_time"].dt.day_name()
df["hour"] = df["start_time"].dt.hour

# Tampilkan beberapa baris data untuk dicek
print("\nHasil FE preprocessing:")
print(df[["id", "start_time", "year", "month", "day", "weekday", "hour"]].head(5))


Hasil FE preprocessing:
     id          start_time  year  month  day   weekday  hour
0  1617 2019-03-09 19:55:00  2019      3    9  Saturday    19
1  1297 2019-03-09 19:54:00  2019      3    9  Saturday    19
2  1394 2019-03-09 18:56:00  2019      3    9  Saturday    18
3  1120 2019-03-09 12:28:00  2019      3    9  Saturday    12
4  2053 2019-03-08 18:25:00  2019      3    8    Friday    18


In [9]:
# Lowercase string
# Ambil semua kolom bertipe object (string/kategorikal)
string_cols = df.select_dtypes(include=["object"]).columns

# Ubah semua kolom string ke lowercase + strip
for col in string_cols:
    df[col] = df[col].str.lower().str.strip()

# Tampilkan beberapa baris data untuk dicek
print("\nContoh data setelah preprocessing:")
print(df.head(5))


Contoh data setelah preprocessing:
     id                                   date  mode  \
0  1617  2019/03/09 20:45 s/d 2019/03/09 19:55  bike   
1  1297  2019/03/09 19:55 s/d 2019/03/09 19:54  food   
2  1394  2019/03/09 19:54 s/d 2019/03/09 18:56  shop   
3  1120  2019/03/09 18:56 s/d 2019/03/09 12:28  food   
4  2053  2019/03/09 12:28 s/d 2019/03/08 18:25   car   

                                         from_alamat   from_kelurahan  \
0                    gang ikhwan no.16,  sungai jawi      darat sekip   
1  neo shabu-shabu steak & shake, johar, jl. joha...  sungai bangkong   
2             alfamart pontianak mall, jl teuku umar      darat sekip   
3  parklife, jl. karimata no.64, sungai bangkong,...          mariana   
4        jl. tabrani ahmad no.12,  sungai jawi dalam         pal lima   

    from_kecamatan               from_latlng  \
0   pontianak kota     -0,0303277,109,297753   
1   pontianak kota       -0,02861,109,329253   
2   pontianak kota    -0,0301863,109,3356331

In [10]:
# Ngatur Birthdate
# Customer Birthdate
if "customer_birthdate" in df.columns:
    df["customer_birthdate"] = pd.to_datetime(df["customer_birthdate"], errors="coerce")

    df["customer_age"] = df["start_time"].dt.year - df["customer_birthdate"].dt.year

    df.loc[
        (df["start_time"].dt.month < df["customer_birthdate"].dt.month) |
        ((df["start_time"].dt.month == df["customer_birthdate"].dt.month) & 
         (df["start_time"].dt.day < df["customer_birthdate"].dt.day)),
        "customer_age"
    ] -= 1

# Driver Birthdate 
if "driver_birthdate" in df.columns:
    df["driver_birthdate"] = pd.to_datetime(df["driver_birthdate"], errors="coerce")

    df["driver_age"] = df["start_time"].dt.year - df["driver_birthdate"].dt.year

    df.loc[
        (df["start_time"].dt.month < df["driver_birthdate"].dt.month) |
        ((df["start_time"].dt.month == df["driver_birthdate"].dt.month) & 
         (df["start_time"].dt.day < df["driver_birthdate"].dt.day)),
        "driver_age"
    ] -= 1

# --- Contoh output ---
print("\nContoh data umur customer & driver:")
print(df[["id", "customer_birthdate", "customer_age", "driver_birthdate", "driver_age"]].head(10))


Contoh data umur customer & driver:
     id customer_birthdate  customer_age driver_birthdate  driver_age
0  1617         1994-02-05            25       1997-03-24          21
1  1297         2004-04-22            14       1976-07-26          42
2  1394         2000-01-07            19       1985-12-28          33
3  1120         1987-08-02            31       1993-06-10          25
4  2053         2004-01-23            15       1988-05-02          30
5  1574         1994-10-31            24       1992-07-12          26
6  1882         1988-04-12            30       1984-01-06          35
7  1958         1992-02-27            27       1991-12-21          27
8  2009         2004-04-01            14       1991-12-21          27
9  1671         1985-03-28            33       1995-01-12          24


In [11]:
# Simpan hasil preprocessing ke file parquet
df.to_parquet("output_preprocessed.parquet", index=False)

print("\nPreprocessing selesai. Data tersimpan di file: output_preprocessed.parquet")


Preprocessing selesai. Data tersimpan di file: output_preprocessed.parquet


In [12]:
# Simpan hasil preprocessing ke file CSV
df.to_csv("output_preprocessed.csv", index=False)

print("\nPreprocessing selesai. Data tersimpan di file: output_preprocessed.csv")


Preprocessing selesai. Data tersimpan di file: output_preprocessed.csv


In [13]:
# MULAI ANALISIS DATA

In [14]:
# 1. Baca hasil preprocessing
df = pd.read_parquet("output_preprocessed.parquet")

print("\nInfo Dataset untuk Analisis")
print("Jumlah baris:", len(df))
print("Jumlah kolom:", len(df.columns))
print("Kolom:\n", df.columns.tolist())


Info Dataset untuk Analisis
Jumlah baris: 1017
Jumlah kolom: 38
Kolom:
 ['id', 'date', 'mode', 'from_alamat', 'from_kelurahan', 'from_kecamatan', 'from_latlng', 'to_alamat', 'to_kelurahan', 'to_kecamatan', 'to_latlng', 'distance', 'amount_delivery', 'amount_merchant', 'transaction_amount_total', 'customer_id', 'customer_gender', 'customer_birthdate', 'driver_id', 'driver_gender', 'driver_birthdate', 'kendaraan_jenis', 'kendaraan_merk', 'merchant_id', 'merchant_name', 'merchant_category', 'customer_gender_encoded', 'kendaraan_jenis_encoded', 'start_time', 'end_time', 'duration_minutes', 'year', 'month', 'day', 'weekday', 'hour', 'customer_age', 'driver_age']


In [15]:
# 1. Berapa banyak total transaksi?
total_transaksi = len(df)

# Transaksi minimum 
min_transaksi = df["amount_delivery"].min()
mode_min = df.loc[df["amount_delivery"] == min_transaksi, "mode"].unique().tolist()

# Transaksi maksimum 
max_transaksi = df["amount_delivery"].max()
mode_max = df.loc[df["amount_delivery"] == max_transaksi, "mode"].unique().tolist()

print(f"Total transaksi: {total_transaksi}")
print(f"Transaksi terkecil: Rp {min_transaksi:,.2f}, Mode: {', '.join(mode_min)}")
print(f"Transaksi terbesar: Rp {max_transaksi:,.2f}, Mode: {', '.join(mode_max)}")

Total transaksi: 1017
Transaksi terkecil: Rp 500.00, Mode: bike
Transaksi terbesar: Rp 32,400.00, Mode: car


In [16]:
# 2. Berapa rata-rata total transaksi?
avg_transaction = df["transaction_amount_total"].mean()
print(f"Rata-rata transaksi: Rp {avg_transaction:,.2f}")

Rata-rata transaksi: Rp 36,206.98


In [17]:
# 3. Bagaimana distribusi mode layanan?
mode_distribution = df["mode"].value_counts().to_frame("Jumlah")
mode_distribution["Persentase (%)"] = (df["mode"].value_counts(normalize=True) * 100).round(2)
print("Distribusi mode layanan:\n", mode_distribution)

Distribusi mode layanan:
       Jumlah  Persentase (%)
mode                        
bike     325           31.96
food     265           26.06
car      242           23.80
shop     185           18.19


In [18]:
# 4. Bagaimana distribusi jenis kelamin driver berdasarkan order?
driver_gender_distribution = df["driver_gender"].value_counts().to_frame("Jumlah")
driver_gender_distribution["Persentase (%)"] = (df["driver_gender"].value_counts(normalize=True) * 100).round(2)
print("Distribusi jenis kelamin driver:\n", driver_gender_distribution)

Distribusi jenis kelamin driver:
                Jumlah  Persentase (%)
driver_gender                        
l                 728           71.58
p                 289           28.42


In [19]:
# 5. Bagaimana distribusi jenis kelamin customer berdasarkan order?
customer_gender_distribution = df["customer_gender"].value_counts().to_frame("Jumlah")
customer_gender_distribution["Persentase (%)"] = (df["customer_gender"].value_counts(normalize=True) * 100).round(2)
print("Distribusi jenis kelamin customer:\n", customer_gender_distribution)

Distribusi jenis kelamin customer:
                  Jumlah  Persentase (%)
customer_gender                        
p                   529           52.02
l                   488           47.98


In [20]:
# 6. Bagaimana distribusi jenis kendaraan?
kendaraan_distribution = df["kendaraan_jenis"].value_counts().to_frame("Jumlah")
kendaraan_distribution["Persentase (%)"] = (df["kendaraan_jenis"].value_counts(normalize=True) * 100).round(2)
print("Distribusi jenis kendaraan driver:\n", kendaraan_distribution)

Distribusi jenis kendaraan driver:
                  Jumlah  Persentase (%)
kendaraan_jenis                        
motor               775            76.2
mobil               242            23.8


In [21]:
# 7. Apa merk kendaraan driver yang paling banyak muncul?
for jenis in df["kendaraan_jenis"].dropna().unique():
    d = df[df["kendaraan_jenis"] == jenis]["kendaraan_merk"].value_counts().rename_axis('kendaraan_merk').reset_index(name='Jumlah')
    d["Persentase (%)"] = (d["Jumlah"] / d["Jumlah"].sum() * 100).round(2)
    print(f"\nDistribusi merk kendaraan untuk {jenis}:\n", d)


Distribusi merk kendaraan untuk motor:
   kendaraan_merk  Jumlah  Persentase (%)
0            bmw     185           23.87
1          honda     146           18.84
2       kawasaki     122           15.74
3            tvs     121           15.61
4         yamaha     117           15.10
5         suzuki      84           10.84

Distribusi merk kendaraan untuk mobil:
   kendaraan_merk  Jumlah  Persentase (%)
0          honda      64           26.45
1            kia      35           14.46
2     mitsubishi      33           13.64
3            bmw      31           12.81
4         toyota      28           11.57
5           audi      26           10.74
6          volvo      25           10.33


In [22]:
# 8. Apa kategori merchant yang paling banyak dipilih? (exclude 'unknown')
merchant_filtered = df[df["merchant_name"].str.lower() != "unknown"]
merchant_category_distribution = merchant_filtered["merchant_category"].value_counts().to_frame("Jumlah")
merchant_category_distribution["Persentase (%)"] = (merchant_filtered["merchant_category"].value_counts(normalize=True) * 100).round(2)
print("Distribusi kategori merchant yang paling banyak dipilih:\n", merchant_category_distribution)

Distribusi kategori merchant yang paling banyak dipilih:
                    Jumlah  Persentase (%)
merchant_category                        
toko/swalayan         185           41.11
cafe                  113           25.11
warung makan           76           16.89
restaurant             42            9.33
jajanan                34            7.56


In [23]:
# 9. Distribusi jenis kelamin customer per mode layanan
gender_mode_distribution = (df.groupby(["mode", "customer_gender"]).size().reset_index(name="Jumlah"))
gender_mode_distribution["Persentase (%)"] = gender_mode_distribution.groupby("mode")["Jumlah"].transform(lambda x: (x / x.sum() * 100).round(2))
print("Distribusi jenis kelamin customer per mode layanan:\n", gender_mode_distribution)

Distribusi jenis kelamin customer per mode layanan:
    mode customer_gender  Jumlah  Persentase (%)
0  bike               l     145           44.62
1  bike               p     180           55.38
2   car               l     121           50.00
3   car               p     121           50.00
4  food               l     129           48.68
5  food               p     136           51.32
6  shop               l      93           50.27
7  shop               p      92           49.73


In [24]:
# 10. Merchant mana yang menerima order terbanyak (exclude 'unknown')?
merchant_filtered = df[df["merchant_name"].str.lower() != "unknown"]
merchant_counts = merchant_filtered["merchant_name"].value_counts().to_frame("Jumlah")
merchant_counts["Persentase (%)"] = (merchant_filtered["merchant_name"].value_counts(normalize=True) * 100).round(2)
merchant_amount = merchant_filtered.groupby("merchant_name")["amount_merchant"].sum().to_frame("Total_Transaksi")
merchant_summary = merchant_counts.join(merchant_amount)
top_3 = merchant_summary.head(3)
bottom_3 = merchant_summary.tail(3)
print("Top 3 merchant:\n", top_3)
print("\nBottom 3 merchant:\n", bottom_3)

Top 3 merchant:
                        Jumlah  Persentase (%)  Total_Transaksi
merchant_name                                                 
alfamart siantan hulu      27            6.00          1677000
indomaret teuku umar       26            5.78          1772500
indomaret kota baru        24            5.33          1504500

Bottom 3 merchant:
                         Jumlah  Persentase (%)  Total_Transaksi
merchant_name                                                  
siobi pontianak              2            0.44            60000
kings' kitchen and bar       1            0.22            31000
pondok rasa                  1            0.22            76000


In [25]:
# 11.Bagaimana tren transaksi per tahun (year)?
total_amount = df.groupby("year")["transaction_amount_total"].sum().to_frame("Total Transaksi")
jumlah_transaksi = df.groupby("year").size().to_frame("Jumlah Transaksi")
transaksi_per_tahun = jumlah_transaksi.join(total_amount).sort_index()
print("Tren transaksi per tahun:\n", transaksi_per_tahun)
jumlah_per_mode = df.groupby(["year", "mode"]).size().unstack(fill_value=0).sort_index()
print("\nJumlah transaksi per tahun per mode:\n", jumlah_per_mode)

Tren transaksi per tahun:
       Jumlah Transaksi  Total Transaksi
year                                   
2018               752         28612700
2019               265          8209800

Jumlah transaksi per tahun per mode:
 mode  bike  car  food  shop
year                       
2018   224  176   202   150
2019   101   66    63    35


In [26]:
# 12.Bagaimana tren transaksi per bulan (month)?
jumlah_transaksi = df.groupby("month").size().to_frame("Jumlah Transaksi")
total_amount = df.groupby("month")["transaction_amount_total"].sum().to_frame("Total Transaksi")
transaksi_per_bulan = jumlah_transaksi.join(total_amount)
print("Tren transaksi per bulan:\n", transaksi_per_bulan)

Tren transaksi per bulan:
        Jumlah Transaksi  Total Transaksi
month                                   
1                   128          4194900
2                   100          3093000
3                    37           921900
9                   101          3509800
10                  146          4595900
11                  127          3666700
12                  378         16840300


In [27]:
# 13. Hari apa yang memiliki transaksi terbanyak?
jumlah_transaksi = df.groupby("weekday").size().to_frame("Jumlah Transaksi")
total_amount = df.groupby("weekday")["transaction_amount_total"].sum().to_frame("Total Transaksi")
transaksi_per_hari = jumlah_transaksi.join(total_amount).sort_values("Jumlah Transaksi", ascending=False)
print("Tren transaksi per hari (weekday):\n", transaksi_per_hari)

Tren transaksi per hari (weekday):
            Jumlah Transaksi  Total Transaksi
weekday                                     
tuesday                 164          6017800
monday                  152          5944900
saturday                148          4984500
thursday                148          5206400
wednesday               146          5297900
sunday                  139          5322500
friday                  120          4048500


In [28]:
# 14. Pada jam berapa (hour) transaksi paling banyak terjadi?
jumlah_transaksi = df.groupby("hour").size().to_frame("Jumlah Transaksi")
total_amount = df.groupby("hour")["transaction_amount_total"].sum().to_frame("Total Transaksi")
summary_per_hour = jumlah_transaksi.join(total_amount)
summary_sorted = summary_per_hour.sort_values("Jumlah Transaksi", ascending=False)
top_10_jam = summary_sorted.head(10)
bottom_3_jam = summary_sorted.tail(3)
print("Top 3 jam dengan jumlah transaksi terbanyak:\n", top_10_jam)
print("\nBottom 3 jam dengan jumlah transaksi tersedikit:\n", bottom_3_jam)

Top 3 jam dengan jumlah transaksi terbanyak:
       Jumlah Transaksi  Total Transaksi
hour                                   
13                  75          2708200
14                  71          2429200
8                   66          2638700
6                   63          2098200
7                   63          1975000
12                  61          2359800
9                   61          2096100
22                  61          2238400
16                  60          2405200
17                  59          3128400

Bottom 3 jam dengan jumlah transaksi tersedikit:
       Jumlah Transaksi  Total Transaksi
hour                                   
20                  55          2037000
10                  49          2122700
21                  42          1192300


In [29]:
# 15. Apakah ada perbedaan jumlah order antara hari kerja vs akhir pekan?
df['day_type'] = df['weekday'].str.lower().apply(lambda x: 'Akhir Pekan' if x in ['saturday', 'sunday'] else 'Hari Kerja')
order_count = df['day_type'].value_counts()
order_percent = (order_count / order_count.sum() * 100).round(2)
summary = pd.DataFrame({'Jumlah Order': order_count, 'Persentase (%)': order_percent})
summary_mode_daytype = df.groupby(['mode', 'day_type']).size().unstack(fill_value=0)
summary_mode_daytype['Total'] = summary_mode_daytype.sum(axis=1)
summary_mode_daytype['% Hari Kerja'] = (summary_mode_daytype['Hari Kerja'] / summary_mode_daytype['Total'] * 100).round(2)
summary_mode_daytype['% Akhir Pekan'] = (summary_mode_daytype['Akhir Pekan'] / summary_mode_daytype['Total'] * 100).round(2)
summary_mode_daytype_ringkas = summary_mode_daytype[['Hari Kerja', 'Akhir Pekan', 'Total', '% Hari Kerja', '% Akhir Pekan']]
print("Jumlah order Hari Kerja vs Akhir Pekan:\n", summary)
print("\nDistribusi order berdasarkan mode dan day_type (ringkas):\n", summary_mode_daytype_ringkas)

Jumlah order Hari Kerja vs Akhir Pekan:
              Jumlah Order  Persentase (%)
day_type                                 
Hari Kerja            730           71.78
Akhir Pekan           287           28.22

Distribusi order berdasarkan mode dan day_type (ringkas):
 day_type  Hari Kerja  Akhir Pekan  Total  % Hari Kerja  % Akhir Pekan
mode                                                                 
bike             229           96    325         70.46          29.54
car              172           70    242         71.07          28.93
food             193           72    265         72.83          27.17
shop             136           49    185         73.51          26.49


In [30]:
# 16. Tren rata-rata distance per bulan
avg_distance_per_bulan = df.groupby("month")["distance"].mean().sort_index()
print("Rata-rata distance per bulan:\n", avg_distance_per_bulan.round(2))

Rata-rata distance per bulan:
 month
1     13.12
2      7.32
3      6.45
9      7.45
10    12.36
11     7.64
12    17.33
Name: distance, dtype: float64


In [31]:
# 17. Apakah durasi lebih lama pada jam sibuk (pagi/sore)?
def categorize_rush_hour(hour):
    if 6 <= hour <= 9:
        return 'Pagi Sibuk'
    elif 16 <= hour <= 19:
        return 'Sore Sibuk'
    else:
        return 'Non Sibuk'
df['rush_hour'] = df['hour'].apply(categorize_rush_hour)
# Hitung rata-rata durasi dan jumlah order per kategori
summary = df.groupby('rush_hour')['duration_minutes'].agg(avg_duration_minutes='mean',total_orders='count').round(2)
def menit_ke_jammenit(menit):
    jam = int(menit // 60)
    sisa_menit = int(menit % 60)
    return f"{jam} jam {sisa_menit} menit"
summary['avg_duration_formatted'] = summary['avg_duration_minutes'].apply(menit_ke_jammenit)
print(summary.reset_index())

    rush_hour  avg_duration_minutes  total_orders avg_duration_formatted
0   Non Sibuk                263.26           531         4 jam 23 menit
1  Pagi Sibuk                205.81           252         3 jam 25 menit
2  Sore Sibuk                292.86           233         4 jam 52 menit


In [32]:
# 18. Berapa rata-rata transaksi di pagi, siang, sore, malam?
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Pagi"
    elif 12 <= hour < 15:
        return "Siang"
    elif 15 <= hour < 19:
        return "Sore"
    else:
        return "Malam"
df["time_of_day"] = df["hour"].apply(get_time_of_day)
avg_amount = df.groupby("time_of_day")["transaction_amount_total"].mean()
count_total = df.groupby("time_of_day").size()
count_mode = df.groupby(["time_of_day", "mode"]).size().unstack(fill_value=0)
summary = pd.concat(
    [avg_amount.rename("rata-rata transaksi"), count_total.rename("jumlah transaksi")],
    axis=1
).join(count_mode)
print("Ringkasan transaksi berdasarkan time_of_day:\n", summary)

Ringkasan transaksi berdasarkan time_of_day:
              rata-rata transaksi  jumlah transaksi  bike  car  food  shop
time_of_day                                                              
Malam               35051.162791               215    73   53    53    36
Pagi                36173.888889               360   109   93    89    69
Siang               36218.357488               207    69   47    54    37
Sore                37305.106383               235    74   49    69    43


In [33]:
# 19. Apakah order lebih banyak dilakukan di awal bulan atau akhir bulan?
df['month_period'] = df['day'].apply(lambda x: 'Awal Bulan' if x <= 15 else 'Akhir Bulan')
order_count = df['month_period'].value_counts()
order_percent = (order_count / order_count.sum() * 100).round(2)
summary_total = pd.DataFrame({'Jumlah Order': order_count, 'Persentase (%)': order_percent})
summary_mode = df.groupby(['mode', 'month_period']).size().unstack(fill_value=0)
summary_mode['Total'] = summary_mode.sum(axis=1)
summary_mode['% Awal Bulan'] = (summary_mode['Awal Bulan'] / summary_mode['Total'] * 100).round(2)
summary_mode['% Akhir Bulan'] = (summary_mode['Akhir Bulan'] / summary_mode['Total'] * 100).round(2)
summary_mode = summary_mode[['Awal Bulan', 'Akhir Bulan', 'Total', '% Awal Bulan', '% Akhir Bulan']]
print("Jumlah order Awal vs Akhir Bulan:\n", summary_total)
print("\nDistribusi order berdasarkan mode dan periode bulan:\n", summary_mode)

Jumlah order Awal vs Akhir Bulan:
               Jumlah Order  Persentase (%)
month_period                              
Akhir Bulan            641           63.03
Awal Bulan             376           36.97

Distribusi order berdasarkan mode dan periode bulan:
 month_period  Awal Bulan  Akhir Bulan  Total  % Awal Bulan  % Akhir Bulan
mode                                                                     
bike                 159          166    325         48.92          51.08
car                   85          157    242         35.12          64.88
food                  79          186    265         29.81          70.19
shop                  53          132    185         28.65          71.35


In [34]:
# 20. Berapa rata-rata jumlah orderan per driver per bulan? 
transactions_per_driver_month = df.groupby(['driver_id', 'month']).size().reset_index(name='num_transactions')
avg_transactions_per_driver = transactions_per_driver_month.groupby('driver_id')['num_transactions'].mean().round(2)
avg_transactions_sorted = avg_transactions_per_driver.sort_values(ascending=False)
print("Rata-rata jumlah transaksi per driver per bulan (dari paling banyak ke paling sedikit):")
print("Driver dengan transaksi terbanyak:")
print(avg_transactions_sorted.head(3))
print("Driver dengan transaksi tersedikit:")
print(avg_transactions_sorted.tail(3))

Rata-rata jumlah transaksi per driver per bulan (dari paling banyak ke paling sedikit):
Driver dengan transaksi terbanyak:
driver_id
88    7.33
94    6.33
91    5.83
Name: num_transactions, dtype: float64
Driver dengan transaksi tersedikit:
driver_id
80    3.33
76    3.29
92    3.14
Name: num_transactions, dtype: float64


In [35]:
# 21.Kecamatan asal mana yang paling banyak order?
kecamatan_count = df['from_kecamatan'].value_counts()
kecamatan_percent = (kecamatan_count / kecamatan_count.sum() * 100).round(2)
mode_distribution = df.pivot_table(index='from_kecamatan', columns='mode', aggfunc='size', fill_value=0)
summary = pd.DataFrame({
    'Jumlah Order': kecamatan_count,
    'Persentase (%)': kecamatan_percent
})
summary = summary.join(mode_distribution).fillna(0)
print("Jumlah order per kecamatan asal dengan persentase dan distribusi per mode:\n", summary)

Jumlah order per kecamatan asal dengan persentase dan distribusi per mode:
                     Jumlah Order  Persentase (%)  bike  car  food  shop
from_kecamatan                                                         
pontianak kota               284           27.93    49   54   133    48
pontianak selatan            179           17.60    48   31    76    24
pontianak barat              164           16.13    55   41    25    43
pontianak utara              152           14.95    59   35     7    51
pontianak tenggara           151           14.85    64   44    24    19
pontianak timur               87            8.55    50   37     0     0


In [36]:
# 22. Apakah jenis kendaraan memengaruhi rata-rata durasi perjalanan 
avg_duration_per_vehicle = df.groupby('kendaraan_jenis')['duration_minutes'].mean().round(2)
avg_duration_per_vehicle_sorted = avg_duration_per_vehicle.sort_values(ascending=False)
def menit_ke_jammenit(menit):
    jam = int(menit // 60)
    sisa_menit = int(menit % 60)
    return f"{jam} jam {sisa_menit} menit"
avg_duration_formatted = avg_duration_per_vehicle_sorted.apply(menit_ke_jammenit)
summary = pd.DataFrame({
    'Rata-rata Durasi (menit)': avg_duration_per_vehicle_sorted,
    'Rata-rata Durasi (jam:menit)': avg_duration_formatted
})
print("Rata-rata durasi perjalanan per jenis kendaraan:\n", summary)

Rata-rata durasi perjalanan per jenis kendaraan:
                  Rata-rata Durasi (menit) Rata-rata Durasi (jam:menit)
kendaraan_jenis                                                       
mobil                              262.29               4 jam 22 menit
motor                              253.77               4 jam 13 menit


In [37]:
# 23. Apakah ada korelasi antara usia pelanggan dan pilihan layanan?
df_encoded = df.dropna(subset=["customer_age", "mode"]).copy()
df_encoded["mode_encoded"] = df_encoded["mode"].astype("category").cat.codes
corr = df_encoded["customer_age"].corr(df_encoded["mode_encoded"])
print(f"Korelasi usia customer dengan pilihan layanan: {corr:.3f}")

Korelasi usia customer dengan pilihan layanan: 0.027


In [38]:
# 24. Kecamatan tujuan mana yang paling sering dituju?
to_kecamatan_count = df['to_kecamatan'].value_counts().reset_index()
to_kecamatan_count.columns = ['to_kecamatan', 'jumlah_order']
avg_distance = df.groupby('to_kecamatan')['distance'].mean().round(2).reset_index()
avg_distance.columns = ['to_kecamatan', 'rata_rata_distance']
summary = to_kecamatan_count.merge(avg_distance, on='to_kecamatan', how='left')
print("Kecamatan tujuan paling sering dituju (dengan rata-rata jarak):\n", summary)

Kecamatan tujuan paling sering dituju (dengan rata-rata jarak):
          to_kecamatan  jumlah_order  rata_rata_distance
0     pontianak timur           190               15.76
1     pontianak utara           180               11.67
2   pontianak selatan           179               19.89
3     pontianak barat           164                6.36
4      pontianak kota           161                6.17
5  pontianak tenggara           143               14.22


In [39]:
# 25. Rata-rata distance dari masing-masing kecamatan asal?
avg_distance_kecamatan = (df.groupby("from_kecamatan")["distance"].mean().sort_values(ascending=False))
print("Rata-rata jarak tempuh per kacamatan asal:\n", avg_distance_kecamatan)

Rata-rata jarak tempuh per kacamatan asal:
 from_kecamatan
pontianak kota        24.865000
pontianak tenggara     8.385563
pontianak timur        8.132989
pontianak utara        7.783289
pontianak barat        7.446220
pontianak selatan      7.191006
Name: distance, dtype: float64


In [40]:
# 26. Apakah ada perbedaan transaksi antar kecamatan asal?
avg_amount_by_kec = (df.groupby("from_kecamatan")["transaction_amount_total"].mean().sort_values(ascending=False))
print("Rata-rata total transaksi per kecamatan asal:\n", avg_amount_by_kec)

Rata-rata total transaksi per kecamatan asal:
 from_kecamatan
pontianak kota        48753.521127
pontianak selatan     42789.385475
pontianak barat       34482.926829
pontianak utara       32238.157895
pontianak tenggara    26472.847682
pontianak timur        8786.206897
Name: transaction_amount_total, dtype: float64


In [41]:
# 27. Bagaimana pola rute yang paling sering dan jarang terjadi?
df["rute"] = df["from_kecamatan"] + " → " + df["to_kecamatan"]
route_counts = df["rute"].value_counts().reset_index()
route_counts.columns = ["rute", "jumlah"]
print("10 Rute Paling Sering Terjadi:\n", route_counts.head(3))
print("\n10 Rute Paling Jarang Terjadi:\n", route_counts.tail(3).sort_values(by="jumlah"))

10 Rute Paling Sering Terjadi:
                                  rute  jumlah
0    pontianak kota → pontianak barat      53
1     pontianak kota → pontianak kota      49
2  pontianak kota → pontianak selatan      48

10 Rute Paling Jarang Terjadi:
                                     rute  jumlah
35      pontianak timur → pontianak kota      10
34  pontianak timur → pontianak tenggara      12
33     pontianak timur → pontianak utara      14


In [42]:
# 28. Bagaimana pola rute yang paling sering terjadi (kelurahan)?
df["rute"] = df["from_kelurahan"] + " → " + df["to_kelurahan"]
route_counts = df["rute"].value_counts().reset_index()
route_counts.columns = ["rute", "jumlah"]
print("10 Rute Paling Sering Terjadi:\n", route_counts.head(3))
print("\n10 Rute Paling Jarang Terjadi:\n", route_counts.tail(3).sort_values(by="jumlah"))

10 Rute Paling Sering Terjadi:
                              rute  jumlah
0    darat sekip → sungai beliung       8
1  bangka belitung darat → akcaya       7
2          mariana → siantan hulu       7

10 Rute Paling Jarang Terjadi:
                                   rute  jumlah
531           tengah → tambelan sampit       1
532  siantan hilir → benua melayu laut       1
533            tengah → banjar serasan       1


In [43]:
# 29. Apakah kecamatan asal berpengaruh terhadap durasi perjalanan?
df["duration_hours"] = df["duration_minutes"] / 60
avg_duration_by_kec = (df.groupby("from_kecamatan")["duration_hours"].mean().round(2).sort_values(ascending=False))
print("Rata-rata durasi perjalanan (jam) per kecamatan asal:", avg_duration_by_kec)

Rata-rata durasi perjalanan (jam) per kecamatan asal: from_kecamatan
pontianak timur       4.71
pontianak barat       4.67
pontianak tenggara    4.53
pontianak utara       4.26
pontianak kota        3.99
pontianak selatan     3.89
Name: duration_hours, dtype: float64


In [44]:
# 30. Apakah kelurahan asal berpengaruh terhadap durasi perjalanan?
df["duration_hours"] = df["duration_minutes"] / 60
avg_duration_by_kel = (df.groupby("from_kelurahan")["duration_hours"].mean().round(2).sort_values(ascending=False))
print("Rata-rata durasi perjalanan (jam) per kelurahan asal paling lama:\n", avg_duration_by_kel.head(3))
# paling bentar
df["duration_hours"] = df["duration_minutes"] / 60
avg_duration_by_kel = (df.groupby("from_kelurahan")["duration_hours"].mean().round(2).sort_values(ascending=True))
print("\nRata-rata durasi perjalanan (jam) per kelurahan asal paling sebentar:\n", avg_duration_by_kel.head(3))

Rata-rata durasi perjalanan (jam) per kelurahan asal paling lama:
 from_kelurahan
bansir laut    5.98
dalam bugis    5.94
saigon         5.77
Name: duration_hours, dtype: float64

Rata-rata durasi perjalanan (jam) per kelurahan asal paling sebentar:
 from_kelurahan
tambelan sampit    2.24
kota baru          2.55
siantan tengah     2.72
Name: duration_hours, dtype: float64


In [45]:
# 31. Berapa rata-rata usia customer?
avg_customer_age = df["customer_age"].mean().round(2)
print(f"Rata-rata usia customer: {avg_customer_age} tahun")

Rata-rata usia customer: 24.44 tahun


In [46]:
# 32. Berapa rata-rata usia customer berdasarkan mode dan jenis kendaraan?
avg_customer_age_grouped = (df.groupby(["mode", "kendaraan_jenis"])["customer_age"].mean().round(2) .reset_index())
print("Rata-rata usia customer per mode dan jenis kendaraan:\n", avg_customer_age_grouped)

Rata-rata usia customer per mode dan jenis kendaraan:
    mode kendaraan_jenis  customer_age
0  bike           motor         24.19
1   car           mobil         24.82
2  food           motor         23.76
3  shop           motor         25.37


In [47]:
# 33. Berapa rata-rata usia driver?
avg_driver_age= df["driver_age"].mean().round(2)
print(f"Rata-rata usia driver: {avg_driver_age} tahun")

Rata-rata usia driver: 29.87 tahun


In [48]:
# 34. Berapa rata-rata usia driver berdasarkan mode dan jenis kendaraan?
avg_driver_age_grouped = (df.groupby(["mode", "kendaraan_jenis"])["driver_age"].mean().round(2) .reset_index())
print("Rata-rata usia customer per mode dan jenis kendaraan:\n", avg_driver_age_grouped)

Rata-rata usia customer per mode dan jenis kendaraan:
    mode kendaraan_jenis  driver_age
0  bike           motor       29.70
1   car           mobil       30.29
2  food           motor       29.72
3  shop           motor       29.82


In [49]:
# 35. Bagaiaman distribusi order berdasarkan gender per driver?
unique_driver_by_gender = (df.groupby("driver_gender")["driver_id"].nunique().reset_index().rename(columns={"driver_id": "jumlah_driver"}))
print("\nJumlah driver berdasarkan gender:\n", unique_driver_by_gender)


Jumlah driver berdasarkan gender:
   driver_gender  jumlah_driver
0             l             25
1             p             10


In [50]:
# 36. Driver mana yang memiliki total pendapatan tertinggi dan terendah?
# Tertinggi
driver_income = (df.groupby("driver_id")["amount_delivery"].sum().reset_index().rename(columns={"amount_delivery": "total_income"}))
driver_income = driver_income.sort_values(by="total_income", ascending=False)
driver_income = driver_income.merge(df[["driver_id", "driver_gender"]].drop_duplicates(),on="driver_id",how="left")
print("5 Driver dengan pendapatan tertinggi:\n", driver_income.head(3))
# Terendah
driver_income = (df.groupby("driver_id")["amount_delivery"].sum().reset_index().rename(columns={"amount_delivery": "total_income"}))
driver_income = driver_income.sort_values(by="total_income", ascending=True)
driver_income = driver_income.merge(df[["driver_id", "driver_gender"]].drop_duplicates(),on="driver_id",how="left")
print("\n5 Driver dengan pendapatan terendah:\n", driver_income.head(3))

5 Driver dengan pendapatan tertinggi:
    driver_id  total_income driver_gender
0         96        519600             l
1         97        474000             l
2         99        472800             l

5 Driver dengan pendapatan terendah:
    driver_id  total_income driver_gender
0         80        107500             l
1         92        114000             l
2         76        131000             l


In [51]:
# 37. Bagaimana distribusi total pendapatan berdasarkan gender?
income_by_gender = (df.groupby("driver_gender")["amount_delivery"].sum().reset_index().rename(columns={"amount_delivery": "total_pendapatan"}))
total_all_income = income_by_gender["total_pendapatan"].sum()
income_by_gender["persentase (%)"] = (
(income_by_gender["total_pendapatan"] / total_all_income * 100).round(2))
print("Distribusi total pendapatan berdasarkan gender driver:\n", income_by_gender)

Distribusi total pendapatan berdasarkan gender driver:
   driver_gender  total_pendapatan  persentase (%)
0             l           6089600           71.34
1             p           2446400           28.66


In [52]:
# 38. Apakah pengemudi mobil menghasilkan pendapatan lebih besar dibanding motor?
income_by_vehicle = (df.groupby("kendaraan_jenis")["amount_delivery"].sum().reset_index().rename(columns={"amount_delivery": "total_pendapatan"}).sort_values(by="total_pendapatan", ascending=False))
total_all_income = income_by_vehicle["total_pendapatan"].sum()
income_by_vehicle["persentase (%)"] = ((income_by_vehicle["total_pendapatan"] / total_all_income * 100).round(2))
print("Total pendapatan berdasarkan jenis kendaraan:\n", income_by_vehicle)

Total pendapatan berdasarkan jenis kendaraan:
   kendaraan_jenis  total_pendapatan  persentase (%)
1           motor           4990000           58.46
0           mobil           3546000           41.54


In [53]:
# 39. Korelasi antara usia driver dengan jumlah order yang diterima?
order_count_per_driver = (df.groupby(["driver_id", "driver_age"])["id"].count().reset_index().rename(columns={"id": "jumlah_order"}))
corr_value = order_count_per_driver["driver_age"].corr(order_count_per_driver["jumlah_order"])
print(f"Korelasi antara usia driver dengan jumlah order: {corr_value:.3f}")

Korelasi antara usia driver dengan jumlah order: 0.046


In [54]:
# 40. Apakah driver laki-laki lebih sering membawa jarak jauh dibanding perempuan?
avg_distance_by_gender = df.groupby("driver_gender")["distance"].mean().round(2).reset_index(name="rata_rata_jarak")
print("Rata-rata jarak perjalanan berdasarkan gender driver:\n", avg_distance_by_gender.sort_values("rata_rata_jarak", ascending=False))

Rata-rata jarak perjalanan berdasarkan gender driver:
   driver_gender  rata_rata_jarak
1             p            15.09
0             l            11.49


In [55]:
# 41. Apakah customer yang lebih muda cenderung order lebih sering?
order_count_per_customer = (df.groupby(["customer_id", "customer_age"])["id"].count().reset_index().rename(columns={"id": "jumlah_order"}))
corr_value = order_count_per_customer["customer_age"].corr(order_count_per_customer["jumlah_order"])
print(f"Korelasi usia customer dengan jumlah order: {corr_value:.3f}")

Korelasi usia customer dengan jumlah order: -0.060


In [56]:
# 42. Customer usia berapa yang paling banyak dan paling sedikit melakukan transaksi?
age_counts = df['customer_age'].value_counts()
top_age = age_counts.idxmax()
bottom_age = age_counts.idxmin()
print(f"Usia customer paling banyak transaksi: {top_age} ({age_counts[top_age]} kali)")
print(f"Usia customer paling sedikit transaksi: {bottom_age} ({age_counts[bottom_age]} kali)")

Usia customer paling banyak transaksi: 18 (77 kali)
Usia customer paling sedikit transaksi: 40 (1 kali)


In [57]:
# 43. Bagaimana distribusi total pendapatan per mode layanan (diluar biaya merchant)?
total_all_income = df["amount_delivery"].sum()
income_by_mode = (df.groupby("mode")["amount_delivery"].sum().reset_index().rename(columns={"amount_delivery": "total_pendapatan"}).sort_values(by="total_pendapatan", ascending=False))
income_by_mode["persentase (%)"] = ((income_by_mode["total_pendapatan"] / total_all_income * 100).round(2))
print("Distribusi total pendapatan berdasarkan mode layanan:\n",income_by_mode.to_string(index=False))

Distribusi total pendapatan berdasarkan mode layanan:
 mode  total_pendapatan  persentase (%)
 car           3546000           41.54
food           2185000           25.60
shop           1554000           18.21
bike           1251000           14.66


In [58]:
# 44. Berapa rata-rata total pemasukkan driver per order?
avg_transaction = df["amount_delivery"].mean()
print(f"Rata-rata transaksi: Rp {avg_transaction:,.2f}")

Rata-rata transaksi: Rp 8,393.31


In [59]:
# 45. Berapa rata-rata total pengeluaran customer untuk membeli barang per order?
avg_transaction = df.loc[df["amount_merchant"] != 0, "amount_merchant"].mean()
print(f"Rata-rata transaksi barang/makanan: Rp {avg_transaction:,.2f}")

Rata-rata transaksi barang/makanan: Rp 62,858.89


In [60]:
# 46. Berapa rata-rata total pemasukkan driver per order berdasarkan jenis kendaraan?
avg_income_by_vehicle = (df.groupby("kendaraan_jenis")["amount_delivery"].mean().round(2).reset_index().rename(columns={"amount_delivery": "rata_rata_pemasukan"}))
print("Rata-rata pemasukan driver per order berdasarkan jenis kendaraan:\n",avg_income_by_vehicle)

# Bagaimana rata - rata pendapatan jika hanya layanan antar?
df_bc = df[df["mode"].isin(["bike", "car"])]
avg_delivery_fee = df_bc.groupby("mode")["amount_delivery"].mean().round(2).reset_index()
avg_delivery_fee.rename(columns={"amount_delivery": "avg_delivery_fee"}, inplace=False)
print("\nRata-rata biaya antar per mode layanan (Bike / Car):\n",avg_delivery_fee)

Rata-rata pemasukan driver per order berdasarkan jenis kendaraan:
   kendaraan_jenis  rata_rata_pemasukan
0           mobil             14652.89
1           motor              6438.71

Rata-rata biaya antar per mode layanan (Bike / Car):
    mode  amount_delivery
0  bike          3849.23
1   car         14652.89


In [61]:
# 47. Yang mana jarak untuk bike dan car paling jauh dan dekat?
# Paling jauh
df_bc = df[df["mode"].isin(["bike", "car"])]
longest_trip_by_mode = (df_bc.loc[df_bc["distance"] == df_bc.groupby("mode")["distance"].transform("max")][["mode", "from_kecamatan", "to_kecamatan", "distance", "amount_delivery"]].sort_values(by="distance", ascending=False))
print("Perjalanan paling jauh per mode layanan (Bike / Car):\n",longest_trip_by_mode)
# Paling dekat
shortest_trip_by_mode = (df_bc.loc[df_bc["distance"] == df_bc.groupby("mode")["distance"].transform("min")][["mode", "from_kecamatan", "to_kecamatan", "distance", "amount_delivery"]].sort_values(by="distance", ascending=True))
print("\nPerjalanan paling dekat per mode layanan (Bike / Car):\n",shortest_trip_by_mode)

Perjalanan paling jauh per mode layanan (Bike / Car):
       mode      from_kecamatan        to_kecamatan  distance  amount_delivery
1003  bike     pontianak utara  pontianak tenggara     17.92             9500
364    car  pontianak tenggara     pontianak timur     15.90            32400

Perjalanan paling dekat per mode layanan (Bike / Car):
      mode   from_kecamatan     to_kecamatan  distance  amount_delivery
578  bike   pontianak kota  pontianak utara      2.09              500
426   car  pontianak timur  pontianak timur      2.51             3600


In [62]:
# 47. Distribusi jenis kelamin customer per mode layanan
gender_mode_distribution = (df.groupby(["mode", "customer_gender"]).size().reset_index(name="Jumlah"))
gender_mode_distribution["Persentase (%)"] = gender_mode_distribution.groupby("mode")["Jumlah"].transform(lambda x: (x / x.sum() * 100).round(2))
print("Distribusi jenis kelamin customer per mode layanan:\n", gender_mode_distribution)

Distribusi jenis kelamin customer per mode layanan:
    mode customer_gender  Jumlah  Persentase (%)
0  bike               l     145           44.62
1  bike               p     180           55.38
2   car               l     121           50.00
3   car               p     121           50.00
4  food               l     129           48.68
5  food               p     136           51.32
6  shop               l      93           50.27
7  shop               p      92           49.73


In [63]:
# Rute kecamatan mana yang paling sering ditempuh dengan motor vs mobil?
# rata-rata transaksi per time_of_day
avg_amount = df.groupby("time_of_day")["transaction_amount_total"].mean()

# jumlah transaksi total
count_total = df.groupby("time_of_day").size()

# jumlah transaksi per mode (misalnya GoRide, GoCar, dsb.)
count_mode = df.groupby(["time_of_day", "mode"]).size().unstack(fill_value=0)

# gabung semua ringkasan
summary = pd.concat(
    [avg_amount.rename("rata-rata transaksi"), 
     count_total.rename("jumlah transaksi")], 
    axis=1
)

summary = summary.join(count_mode)

print(summary)

             rata-rata transaksi  jumlah transaksi  bike  car  food  shop
time_of_day                                                              
Malam               35051.162791               215    73   53    53    36
Pagi                36173.888889               360   109   93    89    69
Siang               36218.357488               207    69   47    54    37
Sore                37305.106383               235    74   49    69    43


In [64]:
# 27. Bagaimana pola rute yang paling sering dipesan oleh cutomer?
df["rute"] = df["from_kecamatan"] + " → " + df["to_kecamatan"]
route_table = (df.pivot_table(index="rute", columns="kendaraan_jenis", values="id", aggfunc="count", fill_value=0)
                 .assign(total_order=lambda x: x.sum(axis=1))
                 .sort_values("total_order", ascending=False)
                 .head(30))
print(route_table)

kendaraan_jenis                          mobil  motor  total_order
rute                                                              
pontianak kota → pontianak barat             8     45           53
pontianak kota → pontianak kota             10     39           49
pontianak kota → pontianak selatan          10     38           48
pontianak kota → pontianak timur            10     37           47
pontianak kota → pontianak utara             8     38           46
pontianak kota → pontianak tenggara          8     33           41
pontianak selatan → pontianak kota           5     35           40
pontianak utara → pontianak timur           10     29           39
pontianak tenggara → pontianak selatan      10     27           37
pontianak selatan → pontianak utara          7     28           35
pontianak barat → pontianak barat           10     25           35
pontianak barat → pontianak tenggara         8     22           30
pontianak selatan → pontianak timur          9     21         

In [65]:
# Hitung omzet per usia dan mode
age_mode_income = df.groupby(["customer_age", "mode"])["transaction_amount_total"].sum().reset_index()

# Cari kelompok usia dengan omzet tertinggi per mode
top_age_mode = age_mode_income.sort_values("transaction_amount_total", ascending=False).groupby("mode").head(5)

print("Kelompok usia dengan omzet tertinggi per mode layanan:\n", top_age_mode)


Kelompok usia dengan omzet tertinggi per mode layanan:
     customer_age  mode  transaction_amount_total
7             14  food                   1645500
74            31  food                   1578500
23            18  food                   1419500
19            17  food                   1311000
24            18  shop                   1168500
35            21  food                   1047500
20            17  shop                    944000
83            33  shop                    907000
75            31  shop                    888500
71            30  shop                    789500
22            18   car                    274800
81            33   car                    273600
73            31   car                    273600
6             14   car                    268800
18            17   car                    222000
21            18  bike                    101500
72            31  bike                     92500
45            24  bike                     90500
17           

In [66]:
# Jumlah transaksi per bulan dan mode
jumlah_per_mode = df.pivot_table(index="month", columns="mode", values="id", aggfunc="count", fill_value=0)

# Total transaksi (rupiah) per bulan dan mode
total_per_mode = df.pivot_table(index="month", columns="mode", values="transaction_amount_total", aggfunc="sum", fill_value=0)

transaksi_per_bulan = pd.concat(
    {
        "Jumlah": df.pivot_table(index="month", columns="mode", values="id", aggfunc="count", fill_value=0),
        "Total": df.pivot_table(index="month", columns="mode", values="transaction_amount_total", aggfunc="sum", fill_value=0)
    },
    axis=1
)

print("Tren transaksi per bulan per mode:\n", transaksi_per_bulan)

Tren transaksi per bulan per mode:
       Jumlah                  Total                           
mode    bike  car food shop    bike      car     food     shop
month                                                         
1         43   37   31   17  172500   518400  2372500  1131500
2         44   17   25   14  170500   252000  1700500   970000
3         14   12    7    4   40500   146400   509000   226000
9         43   14   34   10  167000   214800  2422000   706000
10        64   24   41   17  260000   290400  2830000  1215500
11        62   22   26   17  245000   349200  1910000  1162500
12        55  116  101  106  195500  1774800  7096000  7774000


In [67]:
# Jumlah transaksi per bulan dan mode
jumlah_per_mode = df.pivot_table(index="weekday", columns="mode", values="id", aggfunc="count", fill_value=0)

# Total transaksi (rupiah) per bulan dan mode
total_per_mode = df.pivot_table(index="weekday", columns="mode", values="transaction_amount_total", aggfunc="sum", fill_value=0)

transaksi_per_bulan = pd.concat(
    {
        "Jumlah": df.pivot_table(index="weekday", columns="mode", values="id", aggfunc="count", fill_value=0),
        "Total": df.pivot_table(index="weekday", columns="mode", values="transaction_amount_total", aggfunc="sum", fill_value=0)
    },
    axis=1
)

print("Tren transaksi per bulan per mode:\n", transaksi_per_bulan)

Tren transaksi per bulan per mode:
           Jumlah                 Total                          
mode        bike car food shop    bike     car     food     shop
weekday                                                         
friday        45  26   36   13  179000  402000  2554000   913500
monday        48  30   37   37  214000  410400  2526000  2794500
saturday      48  40   37   23  171500  528000  2766500  1518500
sunday        48  30   35   26  202000  474000  2706500  1940000
thursday      42  40   38   28  144500  536400  2495000  2030500
tuesday       44  46   46   28  167000  706800  3272000  1872000
wednesday     50  30   36   30  173000  488400  2520000  2116500


In [68]:
avg_distance_per_bulan = (
    df.groupby(["month", "mode"])["distance"]
      .mean()
      .round(2)
      .unstack(fill_value=0)
      .sort_index()
)

print("Rata-rata distance per bulan per mode:\n", avg_distance_per_bulan)


Rata-rata distance per bulan per mode:
 mode   bike   car   food  shop
month                         
1      8.29  7.50  30.65  5.62
2      8.04  7.90   6.02  6.72
3      6.29  6.71   6.53  6.06
9      8.11  8.12   6.60  6.51
10     8.41  6.55  24.45  6.31
11     8.15  8.40   6.51  6.53
12     7.52  8.09  44.32  6.81
